In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
df_raw = spark.read.table('ipl.s1_raw.ipl_data')
display(df_raw)

In [0]:
# Convert struct to map to enable dynamic field access by team name

df_raw = df_raw.withColumn("players",col("info.players"))

players_map_df = df_raw.selectExpr(
    "match_id",
    "team1",
    "team2",
    "map_from_entries(array(" + ",".join(
        [f"struct('{field}' as key, players.`{field}` as value)" 
         for field in df_raw.select("players").schema[0].dataType.fieldNames()]
    ) + ")) as players_map"
)

df_players = players_map_df.select(
                col("match_id"),col("team1"),col("team2"),
                col("players_map")[col("team1")].alias("team1_playing11"),
                col("players_map")[col("team2")].alias("team2_playing11")
            )

# df_players = df_players.filter("match_id == '20080418_001'")

df_team1 = df_players\
            .select(col('match_id')
                    ,col('team1').alias('team')
                    ,col('team1_playing11').alias('players'))\
            .withColumn("innings_id",concat(col('match_id'), lit("_1")))\
            .withColumn("innings_num",lit(1))
            # .select('team1','team1_playing11')
            
df_team2 = df_players\
            .select(col('match_id')
                    ,col('team2').alias('team')
                    ,col('team2_playing11').alias('players'))\
            .withColumn("innings_id",concat(col('match_id'), lit("_2")))\
            .withColumn("innings_num",lit(2))
            # .select('team2','team2_playing11')

df_players = df_team1.union(df_team2 )

df_players = df_players.select\
                (
                    col("match_id"),
                    col("innings_id").alias("team_id"),
                    col("team"),
                    col("players"),
                    posexplode("players").alias("seq","player")
                )\
                .withColumn("playing11_id",concat(col("team_id"),lit("_"),lpad(col("seq")+1,2,0)))\
                .drop("seq","players")


display(
    df_players
    )



In [0]:
df_players.write.mode("overwrite").saveAsTable("ipl.s2_silver.playing11")